# 01 — Data understanding (Home Credit Default Risk)

Goals: locate CSVs, confirm shapes and dtypes, quantify missing values (sample or full scan), inspect `TARGET` on `application_train`, and skim numeric ranges.

Put competition CSVs under `data/raw/` next to this repo (or pass `resolve_data_dir(preferred=...)`).

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data_understanding import (
    EXPECTED_FILES,
    application_train_specials,
    dtypes_and_columns,
    numeric_ranges_sample,
    resolve_data_dir,
    scan_csv_paths,
    summarize_csv,
)

# Point here if your CSVs live elsewhere:
DATA_DIR = resolve_data_dir(preferred=PROJECT_ROOT / "data" / "raw")
DATA_DIR

## Expected vs available files

In [ ]:
paths = scan_csv_paths(DATA_DIR)

avail = pd.DataFrame(
    {"expected_csv": list(EXPECTED_FILES.values()), "logical_name": list(EXPECTED_FILES.keys())}
)
avail["found"] = avail["logical_name"].map(lambda k: k in paths)
avail

## Table inventory (rows, columns, size)

Missing summaries below default to **`missing="sample"`** (first 100k rows) so large tables stay responsive. Set `missing="full"` inside `summarize_csv` when you want exact missing rates (slow on bureau-level tables).

In [ ]:
inventory_rows = []
for logical_name in sorted(paths):
    p = paths[logical_name]
    summary = summarize_csv(p, missing="sample")
    inventory_rows.append(
        {
            "table": logical_name,
            "rows": summary["rows"],
            "cols": summary["cols"],
            "size_mb": summary["size_mb"],
            "missing_mode": summary["missing_mode"],
        }
    )

inventory = pd.DataFrame(inventory_rows).sort_values("table")
inventory

## Top missing-rate columns (per table, sampled)

In [ ]:
for logical_name in sorted(paths):
    p = paths[logical_name]
    s = summarize_csv(p, missing="sample")
    print("\n", logical_name, "—", p.name)
    display(s["top_missing"])

## Target & IDs on `application_train`

In [ ]:
train_path = paths["application_train"]
application_train_specials(train_path)

## Dtypes & numeric skim (prefix samples)

Use these cells when you want a quick view without scanning entire files.

In [ ]:
logical_name = "application_train"
p = paths[logical_name]

display(dtypes_and_columns(p, nrows=8000).head(40))
numeric_ranges_sample(p, nrows=80_000).head(25)

### Optional: exact missing rates (`missing="full"`)

Uncomment and run on selected tables only — full scans over tens of millions of rows can take noticeable time.

```python
# summarize_csv(paths["application_train"], missing="full")["top_missing"]
```